# 07 可选：番茄叶片八分类交互演示

完成主实验后，加载验证选定的最终 E3 模型，显示上传图片的类别概率与可选 Grad-CAM。安装 `requirements-demo.txt`，再开启 `LAUNCH_DEMO`。

默认不启动服务。演示在本机运行；类别只覆盖本研究八类。模型输出概率不代表病理学诊断置信度，不提供未经核实的防治建议。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请在项目根目录或 notebooks 目录打开本 Notebook。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, load_labels

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")
CLASS_NAMES = [item["label"] for item in load_labels(PROJECT_ROOT)]
assert len(CLASS_NAMES) == 8 and len(set(CLASS_NAMES)) == 8
assert [item["label_index"] for item in LABELS["classes"]] == list(range(8))
print("项目目录：", PROJECT_ROOT)
print("类别数量：", len(CLASS_NAMES), "；随机种子：", BASE["seed"])


项目目录： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project
类别数量： 8 ；随机种子： 42


## 1. 解析最终模型

模型目录自动来自 `selected_runs.e3`，类别与预处理以保存的 checkpoint 为准；示例图片可自行放入 `demo/examples/`，注明来源。


In [2]:
from src.experiments import selected_run

LAUNCH_DEMO = True
demo_run_dir = None
try:
    demo_run_dir = selected_run(PROJECT_ROOT, "E3")
    print("最终八分类模型：", demo_run_dir.name)
except (ValueError, FileNotFoundError) as error:
    print("请先完成 02—04 并保存 E3 选择：", error)
print("启动演示：", LAUNCH_DEMO)


最终八分类模型： e3_efficientnet_b0_full_combined_s42_20261021T065510_c05233
启动演示： True


## 2. 启动本地界面

开启开关后运行本格。勾选热力图才计算 Grad-CAM；先用少量有来源记录的图片确认上传、推理和展示可用，再用于课程展示。


In [5]:
demo = None
if not LAUNCH_DEMO:
    print("演示关闭：准备好后将 LAUNCH_DEMO 设为 True。")
elif demo_run_dir is None:
    print("尚无完成的最终八分类模型。")
else:
    import gradio as gr
    from src.data import build_transforms
    from src.evaluate import load_trained_model, predict_image
    from src.explain import gradcam_image
    from src.utils import get_device

    device = get_device(BASE.get("device", "auto"))
    model, checkpoint = load_trained_model(demo_run_dir, device=device)
    data_config = checkpoint["config"]["data"]
    transform = build_transforms("none", **{key: data_config[key]
                                 for key in ("image_size", "mean", "std") if key in data_config})
    class_names = checkpoint["class_names"]
    display_names = {item["label"]: item.get("display_name", item["label"])
                     for item in LABELS["classes"]}

    def predict_for_demo(image, show_heatmap):
        if image is None:
            return {}, None, "请先上传一张番茄叶片照片。"
        result = predict_image(model, image, transform, class_names, device=device)
        probabilities = {display_names.get(label, label): score
                         for label, score in result["probabilities"].items()}
        overlay = None
        if show_heatmap:
            overlay = gradcam_image(model, checkpoint["model_name"], image, transform,
                                    target_class=result["label_index"], device=device)
        label_text = display_names[result["label"]]
        details = f"**模型预测：{label_text}**，模型输出概率 {result['confidence']:.1%}。"
        details += "\n\n仅供课程研究展示。复杂背景、其他作物或八类之外病害可能导致误判。"
        return probabilities, overlay, details

    example_dir = PROJECT_ROOT / "demo" / "examples"
    example_paths = sorted(path for path in example_dir.glob("*")
                           if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})[:6]
    demo = gr.Interface(
        fn=predict_for_demo,
        inputs=[gr.Image(type="pil", label="上传番茄叶片照片"),
                gr.Checkbox(value=False, label="生成 Grad-CAM 热力图")],
        outputs=[gr.Label(num_top_classes=8, label="八类模型概率"),
                 gr.Image(type="pil", label="Grad-CAM"), gr.Markdown()],
        title="番茄叶片病害八分类：迁移学习课程演示",
        description="模型覆盖七种病害与健康状态；结果用于课程研究展示。",
        examples=[[str(path), False] for path in example_paths] or None,
    )
    import os

    # 让本地访问绕过代理
    os.environ["NO_PROXY"] = "127.0.0.1,localhost,::1"
    os.environ["no_proxy"] = "127.0.0.1,localhost,::1"

    # 在本机启动 Gradio
    demo.launch(
            inline=False,
            share=False,
            server_name="127.0.0.1",
            server_port=7860,
            show_error=True
        )


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

In [ ]:
# 需要停止演示时单独运行本格。
CLOSE_DEMO = False
if CLOSE_DEMO and demo is not None:
    demo.close()
    print("演示已关闭。")


## 展示记录（实际启动后填写）

- 最终模型目录与验证选择依据：待填写。
- 上传、八类概率、Grad-CAM、停止服务的验证情况：待填写。
- 示例来源、失败场景与局限：待填写。
